# Module 4 — Text-to-SQL (exercises)

Grounded in real project data: `learning/days/day5/eda/data/player_stats_2025-26_regular.csv`
(2025-26 regular-season player stats pulled via `nba_api`). Everything runs on DuckDB, already a
repo dependency — no new installs. Companion reading: `workbook.md`.

Core cells are dependency-light (`duckdb`, stdlib). Cell 6 (prompt construction) is optional.


In [ ]:
import duckdb
from pathlib import Path

here = Path.cwd()
root = None
for p in [here, *here.parents]:
    if (p / 'learning/days/day5/eda/data/player_stats_2025-26_regular.csv').exists():
        root = p; break
assert root is not None, 'Run this notebook from inside the statlas repo (any subdir).'

CSV = root / 'learning/days/day5/eda/data/player_stats_2025-26_regular.csv'
con = duckdb.connect()
con.execute(f"CREATE TABLE players AS SELECT * FROM read_csv_auto('{CSV.as_posix()}')")
schema = con.execute("DESCRIBE players").fetchall()
print(f"{len(schema)} columns loaded")
for name, typ, *_ in schema[:12]:
    print(f"  {name:20s} {typ}")
print("  ...")


## Exercise 1 — The schema you'd hand an LLM
Print the full column list + a couple of sample rows. This *is* the schema-linking context a
text-to-SQL prompt needs (DIN-SQL step 1).

In [ ]:
COLUMNS = [r[0] for r in con.execute("DESCRIBE players").fetchall()]
assert 'PLAYER_NAME' in COLUMNS and 'PTS' in COLUMNS and 'GP' in COLUMNS
print(COLUMNS)
con.execute("SELECT PLAYER_NAME, TEAM_ABBREVIATION, GP, PTS, REB, AST FROM players LIMIT 3").df()


## Exercise 2 — Hand-write the gold set
Write **8** (question, SQL) pairs against `players`. Cover: a simple lookup, a simple aggregate,
a plain `ORDER BY ... LIMIT` leaderboard, and **at least one leaderboard with a `HAVING GP >= 58`
qualifier** (the repo's actual reliability rule — see root `CLAUDE.md`). Run each and eyeball the
result before moving on.

`# TODO`: fill in at least 8 entries; the assert checks you did.

In [ ]:
GOLD = [
    ("How many points per game does A.J. Lawson average?",
     "SELECT PTS FROM players WHERE PLAYER_NAME = 'A.J. Lawson'"),
    ("Who leads the league in points per game?",
     "SELECT PLAYER_NAME, PTS FROM players ORDER BY PTS DESC LIMIT 1"),
    ("Top 5 steals leaders with at least 58 games played",
     "SELECT PLAYER_NAME, STL FROM players WHERE GP >= 58 ORDER BY STL DESC LIMIT 5"),
    # TODO: add at least 5 more pairs (assists leaders, a team filter, a per-36-ish ratio,
    # a min-games-qualified 3PT leader, a lookup by team abbreviation, etc.)
]

assert len(GOLD) >= 8, f"Add more gold pairs — you have {len(GOLD)}, need >= 8"
for q, sql in GOLD:
    res = con.execute(sql).fetchall()
    assert len(res) > 0, f"Query returned nothing: {sql}"
print(f"{len(GOLD)} gold pairs, all executable.")


## Exercise 3 — `assert_safe`: the allow-list gate
Implement a SELECT-only validator. It must **reject** every attack string and **accept** every
legitimate query. This is the literal spec for `src/statlas/query/planner.py`'s `assert_safe`.

In [ ]:
import re

ALLOWED_TABLES = {"players"}
ALLOWED_COLUMNS = set(c.upper() for c in COLUMNS)
FORBIDDEN_KEYWORDS = {"INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "ATTACH",
                       "COPY", "PRAGMA", "EXEC", "EXECUTE", "GRANT", "TRUNCATE"}

def assert_safe(sql: str) -> bool:
    """Return True iff `sql` is a single, allow-listed, SELECT-only statement."""
    stripped = sql.strip().rstrip(';')
    # TODO 1: reject if there's a second statement (a bare ';' followed by non-whitespace,
    #         or the original string contains ';' before the very end)
    if ';' in stripped:
        return False
    # TODO 2: reject if it doesn't start with SELECT (case-insensitive)
    if not re.match(r'^\s*SELECT\b', stripped, re.IGNORECASE):
        return False
    # TODO 3: reject if any forbidden keyword appears anywhere (as a whole word)
    tokens = set(re.findall(r'[A-Za-z_]+', stripped.upper()))
    if tokens & FORBIDDEN_KEYWORDS:
        return False
    # TODO 4 (stretch): reject if a referenced table isn't in ALLOWED_TABLES
    #         (a real implementation would parse the FROM/JOIN clauses; a cheap heuristic:
    #          every bare identifier after FROM/JOIN must be allow-listed)
    from_tables = re.findall(r'\bFROM\s+([A-Za-z_]+)|\bJOIN\s+([A-Za-z_]+)', stripped, re.IGNORECASE)
    for a, b in from_tables:
        t = (a or b).lower()
        if t not in ALLOWED_TABLES:
            return False
    return True

ATTACKS = [
    "DROP TABLE players",
    "SELECT * FROM players; DELETE FROM players",
    "SELECT * FROM players -- ; DROP TABLE players",
    "INSERT INTO players VALUES (1,2,3)",
    "SELECT * FROM other_secret_table",
    "ATTACH DATABASE '/etc/passwd' AS x",
]
LEGIT = [g[1] for g in GOLD[:4]]

for a in ATTACKS:
    assert not assert_safe(a), f"FAILED TO BLOCK: {a}"
for l in LEGIT:
    assert assert_safe(l), f"FALSE POSITIVE (blocked a legit query): {l}"
print(f"assert_safe: blocked {len(ATTACKS)}/{len(ATTACKS)} attacks, "
      f"passed {len(LEGIT)}/{len(LEGIT)} legit queries.")


## Exercise 4 — Execution-accuracy scorer
Two SQL strings can be textually different but **execution-equal**. Implement `exec_match` that
compares result *multisets*, order-insensitive, with float tolerance.

In [ ]:
def exec_match(pred_sql: str, gold_sql: str, conn, tol: float = 1e-6) -> bool:
    pred = conn.execute(pred_sql).fetchall()
    gold = conn.execute(gold_sql).fetchall()
    if len(pred) != len(gold):
        return False
    def norm(row):
        return tuple(round(v, 6) if isinstance(v, float) else v for v in row)
    # multiset comparison: sort both (rows are hashable tuples after norm)
    return sorted(map(norm, pred)) == sorted(map(norm, gold))

# same result, different SQL surface form -> should match
gold_sql = "SELECT PLAYER_NAME, STL FROM players WHERE GP >= 58 ORDER BY STL DESC LIMIT 5"
pred_sql = "SELECT PLAYER_NAME AS n, STL AS s FROM players WHERE GP >= 58 ORDER BY 2 DESC LIMIT 5"
# TODO: exec_match compares *values*, not column names/aliases -- confirm this still matches
assert exec_match(pred_sql, gold_sql, con), "String-different, execution-equal queries should match"

# a genuinely wrong query (missing the qualifier) -> should NOT match. STL is a good example:
# without the GP>=58 filter, a 3-game small-sample "leader" (Kadary Richmond) tops the board.
wrong_sql = "SELECT PLAYER_NAME, STL FROM players ORDER BY STL DESC LIMIT 5"
assert not exec_match(wrong_sql, gold_sql, con), "Missing the GP>=58 qualifier should change the result"
print("exec_match: correctly matched an equal-but-different query and rejected a wrong one.")


## Exercise 5 — Schema-linking heuristic
A cheap string/fuzzy function guessing which columns a question references — the thing a model
(or a prompt-construction step) does before generating SQL.

In [ ]:
SYNONYMS = {
    "points": "PTS", "scoring": "PTS", "ppg": "PTS",
    "rebounds": "REB", "boards": "REB",
    "assists": "AST", "dimes": "AST",
    "games": "GP", "games played": "GP",
    "team": "TEAM_ABBREVIATION",
    "player": "PLAYER_NAME", "name": "PLAYER_NAME",
    "steals": "STL", "blocks": "BLK", "turnovers": "TOV",
}

def link(question: str, columns) -> set:
    q = question.lower()
    hits = set()
    for phrase, col in SYNONYMS.items():
        if phrase in q and col in columns:
            hits.add(col)
    # TODO: also catch direct column-name mentions (e.g. "PTS" typed literally)
    for col in columns:
        if col.lower() in q:
            hits.add(col)
    return hits

for q, sql in GOLD:
    linked = link(q, ALLOWED_COLUMNS)
    print(f"{q!r:60s} -> {linked}")
    # a soft check, not a hard assert: at least one column should usually be linked


## Exercise 6 (OPTIONAL) — Prompt construction
Build the actual few-shot prompt string you'd send an LLM, DIN-SQL style: schema + 2 examples +
the new question. No API call — just inspect the string.

In [ ]:
def build_prompt(question, gold_examples, columns, table='players'):
    schema_block = f"Table {table}({', '.join(columns)})"
    examples_block = "\n\n".join(
        f"Q: {q}\nSQL: {sql}" for q, sql in gold_examples[:2]
    )
    return f"""{schema_block}

{examples_block}

Q: {question}
SQL:"""

prompt = build_prompt("Who has the most steals?", GOLD, sorted(ALLOWED_COLUMNS))
print(prompt)


## Math problems
Work these by hand in a markdown cell or on paper (see `workbook.md` for the full problem
statements), then verify Problem 2 (self-consistency arithmetic) here.

In [ ]:
from itertools import product
from math import comb

def majority_correct_prob(k: int, p: float) -> float:
    """P(majority of k i.i.d. Bernoulli(p) trials are 'correct'), k odd."""
    need = k // 2 + 1
    return sum(comb(k, i) * p**i * (1 - p)**(k - i) for i in range(need, k + 1))

for p in (0.6, 0.8):
    single = p
    triple = majority_correct_prob(3, p)
    print(f"p={p}: single-shot={single:.3f}  majority-of-3={triple:.3f}  "
          f"gain={triple - single:+.3f}")

# TODO: at what p does majority-of-3 start losing to single-shot? (hint: try p < 0.5)
assert majority_correct_prob(3, 0.6) > 0.6, "Majority-of-3 should beat single-shot when p=0.6"


## Definition of done
- [ ] 8+ gold pairs, all executable
- [ ] `assert_safe` blocks all 6 attacks, passes all legit queries
- [ ] `exec_match` correctly identifies the string-different/execution-equal pair
- [ ] Math Problem 2 verified in code above
